In [17]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [18]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [19]:
df.shape

(569, 33)

In [20]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


train test split

In [21]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [22]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [26]:
y_train

110    B
543    B
42     M
424    B
248    B
      ..
99     M
288    B
280    M
268    B
82     M
Name: diagnosis, Length: 455, dtype: str

label encoding


In [27]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [28]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [29]:
X_train_tensor

tensor([[-1.2550, -0.4966, -1.2351,  ..., -0.9297, -0.6104,  0.0256],
        [-0.2753,  2.1454, -0.3076,  ..., -0.5390, -0.7062, -1.0721],
        [ 1.3968,  1.3698,  1.4919,  ...,  1.9879,  2.8032,  1.0619],
        ...,
        [ 1.4225,  1.7970,  1.4048,  ...,  1.0633,  0.5477,  0.7042],
        [-0.3724, -0.6827, -0.4112,  ..., -0.8633,  1.1004, -0.7366],
        [ 3.1518,  1.3937,  3.2822,  ...,  2.5447, -0.8947,  1.1324]],
       dtype=torch.float64)

In [30]:
X_train_tensor.shape

torch.Size([455, 30])

In [32]:
y_train_tensor.shape

torch.Size([455])

defining model

In [33]:
class SimpleNN():
    def __init__(self, X):
        self.weights = torch.rand(X.shape[1],1,  dtype=torch.float64, requires_grad= True)
        self.bias = torch.zeros(1,  dtype=torch.float64, requires_grad=True)
    
    def forward(self, X):
        z = torch.matmul(X, self.weights) + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred
    
    def loss_function(self, y_pred, y):
        # Clamp predictions to avoid log(0)
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)
        # Calculate loss
        loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred)).mean()
        return loss


In [35]:
learning_rate = 0.1
epochs = 25

In [39]:
#create model
model = SimpleNN(X_train_tensor)

#define loop 
for epoch in range(epochs):
    
    #forward pass
    y_pred = model.forward(X_train_tensor)
    #loss claculation
    loss = model.loss_function(y_pred, X_train_tensor)
    #backword pass
    loss.backward()
    #parameter
    with torch.no_grad():
        model.weights -= learning_rate*model.weights.grad
        model.bias -= learning_rate*model.bias.grad

    #zero gradient
    model.weights.grad.zero_()
    model.bias.grad.zero_()

    #print loss in each epoch
    print(f"Epoch: {epoch +1}, Loss: {loss.item()}")

Epoch: 1, Loss: 3.880239010329982
Epoch: 2, Loss: 3.7571889952912407
Epoch: 3, Loss: 3.6300061277849136
Epoch: 4, Loss: 3.495852695346134
Epoch: 5, Loss: 3.3556738821243877
Epoch: 6, Loss: 3.2120159523559066
Epoch: 7, Loss: 3.0657260504956145
Epoch: 8, Loss: 2.915970225649513
Epoch: 9, Loss: 2.7615572368299217
Epoch: 10, Loss: 2.6006591983889464
Epoch: 11, Loss: 2.4325678727607563
Epoch: 12, Loss: 2.262640723649602
Epoch: 13, Loss: 2.0934122738894607
Epoch: 14, Loss: 1.922955224914582
Epoch: 15, Loss: 1.7571066424950834
Epoch: 16, Loss: 1.6012616543070783
Epoch: 17, Loss: 1.4535891331868713
Epoch: 18, Loss: 1.3203644558681034
Epoch: 19, Loss: 1.2033873259190864
Epoch: 20, Loss: 1.1040131756363178
Epoch: 21, Loss: 1.0227006508135876
Epoch: 22, Loss: 0.9586504933157663
Epoch: 23, Loss: 0.9098107342004191
Epoch: 24, Loss: 0.873337771209577
Epoch: 25, Loss: 0.8462558776079163


In [46]:
model.weights
model.bias

tensor([-0.1232], dtype=torch.float64, requires_grad=True)

In [47]:
#model evalution
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred = (y_pred > 0.5).float()
    accuracy = (y_pred == y_test_tensor).float().mean()
    print(f'Accuracy: {accuracy.item()}')



Accuracy: 0.517236053943634
